# AnchorFlow Research · V4 ContextWide / V3 matched-budget control
Upload **nguyên folder AnchorFlow_Research** vào MyDrive. Default train **v4 thêm 15 epoch từ v3 best epoch14** (đã học tổng cộng30epoch).

- Chỉ đổi `MODEL_NAME = "v4"` / `"v3"` ở cell1. Data, loss, augmentation, optimizer recipe, evaluator và outputs dùng chung.
- **V3 control** là train tiếp v3 từ cùng checkpoint thêm15epoch; chạy riêng nếu muốn kết luận architecture gain. So v4 với checkpoint v3 cũ đơn thuần còn lẫn ảnh hưởng train thêm.
- Default **một run v4**, không tự chạy hai experiment.
- Metric teacher chỉ train, val/test teacher-free. 1.600train/400val/1.000anonymous test, giữ GT support.
- Parent thật: RMSE **1,04537m**. V4 chưa có accuracy sau train; mục tiêu <0,7m không được bảo đảm.
- Code đã pull remote; augmentation/server mới của S3 giữ độc lập, **không tự bật vào experiment này**.

## 1. Mount, chọn model và đường dẫn

Chọn GPU trước khi Run all. Chỉ sửa MODEL_NAME để đổi v3/v4; mỗi model có run folder riêng, cùng parent checkpoint. Giữ EPOCHS=15.

Colab dùng SSD /content để cache/train, không dùng ổ máy cá nhân. Không xóa dataset/run cũ.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

BUNDLE = Path("/content/drive/MyDrive/AnchorFlow_Research")
DRIVE_DATA = Path("/content/drive/MyDrive/GeoLift_Data")
DRIVE_RUNS = Path("/content/drive/MyDrive/GeoLift_RT_Runs")
WORK = Path("/content/anchorflow_edge_work")  # reuse data cache of v2 if this runtime still has it
CODE = Path("/content/anchorflow_research_code")
MODEL_NAME = "v4"  # "v3" = matched-budget control; only switch this
assert MODEL_NAME in ("v3", "v4")
RUN_NAMES = {"v4": "AnchorFlow_v4_ContextWide_FT15", "v3": "AnchorFlow_v3_Control_FT15"}
RUN_NAME = RUN_NAMES[MODEL_NAME]
EPOCHS = 15
BATCH_SIZE = 2
WORKERS = 2
RUN_EXPORT = True
assert BUNDLE.is_dir(), f"Upload folder to {BUNDLE}"
assert EPOCHS == 15
RUN_ROOT = DRIVE_RUNS / RUN_NAME
RUN_DIR = RUN_ROOT / "metric_kd"
print("Input:", DRIVE_DATA / "teacher_subset_2000")
print("Outputs:", RUN_DIR)
print("Colab SSD free GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))

## 2. Verify và copy bundle nhỏ

Không xóa folder dữ liệu/run cũ. Nếu SHA không khớp, upload lại **trọn bundle mới**, không bỏ qua kiểm tra. Notebook có thể được Colab lưu metadata nên không tự kiểm SHA chính nó.

In [ ]:
manifest = json.loads((BUNDLE / "bundle_manifest.json").read_text(encoding="utf-8"))
assert manifest["default_epochs"] == 15
assert manifest["default_variants"] == ["metric_kd"]
assert manifest["contains_data"] is False and manifest["contains_parent_student_checkpoint"] is True
CODE.mkdir(parents=True, exist_ok=True)
for name, expected in manifest["files"].items():
    assert Path(name).name == name, name
    if name.endswith(".ipynb"):
        continue
    source = BUNDLE / name
    raw = source.read_bytes()
    assert hashlib.sha256(raw).hexdigest() == expected, f"Bundle mismatch: {name}"
    destination = CODE / name
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
shutil.copy2(BUNDLE / "bundle_manifest.json", CODE / "bundle_manifest.json")
os.chdir(CODE)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements.txt")], check=True)
import torch, timm
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
assert tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 4)
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__, "| timm:", timm.__version__)

## 3. Config chung, warm-start từ best v3

encoder_pretrained=false vì **nạp đủ trọng số đã train**, không phải random-init. Parent checkpoint7,53MB có trong folder.

Hai model cùng peak LR encoder2,5e-5 / mọi non-encoder1e-4, warm-up1epoch→cosine15epoch. Loss v3 giữ nguyên, KD0,05→gần0,025. Không thêm range weight, teacher hay augmentation mới.

Warm-start v3→v4 reset optimizer có chủ ý; resume run đang chạy thì khôi phục đầy đủ.

In [ ]:
cfg = json.loads((CODE / "config.json").read_text(encoding="utf-8"))
cfg.update(drive_data=str(DRIVE_DATA), drive_runs=str(DRIVE_RUNS), work=str(WORK),
           run_name=RUN_NAME, model_name=MODEL_NAME, epochs=EPOCHS, batch_size=BATCH_SIZE, workers=WORKERS,
           init_checkpoint=str(CODE / "init_v3_best.pth"))
cfg["architecture"] = {"v4": "AnchorFlow-Edge-v4-ContextWide", "v3": "AnchorFlow-Edge-v3-RefineKD"}[MODEL_NAME]
assert cfg["parent_cumulative_epoch"] == 29
assert cfg["new_lr_ratio"] == 1.0
assert cfg["parent_epoch"] == 14 and cfg["teacher_enabled"] is True
assert cfg["flow_steps"] == 3 and cfg["encoder_pretrained"] is False
assert hashlib.sha256(Path(cfg["init_checkpoint"]).read_bytes()).hexdigest() == cfg["init_sha256"]
CONFIG = CODE / "resolved_config.json"
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
SUBSET = DRIVE_DATA / "teacher_subset_2000"
for name in ("selected_2000_ids.json", "kitti_trainval_2000.tar", cfg["metric_tar"]):
    assert (SUBSET / name).is_file(), f"Thiếu {SUBSET / name}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
snapshot = RUN_ROOT / "source_bundle"
snapshot.mkdir(parents=True, exist_ok=True)
for name in manifest["files"]:
    if name.endswith(".ipynb"):
        continue
    source = CODE / name
    target = snapshot / name
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == hashlib.sha256(source.read_bytes()).hexdigest(), f"Run này có source khác: {name}; dùng RUN_NAME mới"
    else:
        shutil.copy2(source, target)
shutil.copy2(BUNDLE / "bundle_manifest.json", snapshot / "bundle_manifest.json")
print(CONFIG.read_text())

def command(action, extra=()):
    subprocess.run([sys.executable, "-u", str(CODE / "run.py"), action,
                    "--config", str(CONFIG), "--variant", "metric_kd", *extra],
                   cwd=CODE, check=True)

## 4. Contract tests và data gate

Kiểm đủ 2.000 teacher record trong TAR, nhưng chỉ cache/load **1.600 train teacher**. Val/test không có teacher trong batch.

KD dùng `C_cm >= 0.5`, ngoài vùng có GT và **toàn bộ sparse gốc** kể cả sparse bị holdout. Mỗi mẫu train cần ít nhất 1.024 pixel teacher eligible. Nếu gate lỗi, đọc báo cáo; không hạ điều kiện để chạy qua.

Anonymous test lấy từ `GeoLift_Data/test_1000/kitti_test_1000.tar`. Nếu chưa có, helper tải nguồn KITTI official một lần rồi lưu TAR vào Drive. Không dùng 1.000 ảnh này để tính validation RMSE.

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(CODE), "-p", "test*.py", "-v"],
               cwd=CODE, check=True)
command("prepare")
contract = json.loads((WORK / "data_contract.json").read_text())
assert (contract["train"], contract["val"], contract["test"]) == (1600, 400, 1000)
assert contract["validation_teacher_used"] is False
assert contract["effective_kd_confidence_threshold"] == cfg["kd_conf_min"]
assert contract["min_non_gt_teacher_pixels"] >= cfg["min_teacher_pixels"]
shutil.copy2(WORK / "data_contract.json", RUN_ROOT / "data_contract.json")
print(json.dumps(contract, indent=2))

## 5. Inspect đúng mẫu train/val

GT là KITTI depth; metric teacher có thể đã dùng GT khi generate nên không coi teacher-vs-GT là đánh giá độc lập.

In [ ]:
from data import KITTIDataset
import matplotlib.pyplot as plt
import numpy as np

train_data = KITTIDataset(cfg, "train", teacher=True)
val_data = KITTIDataset(cfg, "val", teacher=False)
assert len(train_data) == 1600 and len(val_data) == 400
train_sample, val_sample = train_data[0], val_data[0]
assert "teacher" not in val_sample and "confidence" not in val_sample
for label, sample in (("train", train_sample), ("val", val_sample)):
    names = ["RGB", "Sparse", "GT"] + (["Metric teacher", "C_cm"] if "teacher" in sample else [])
    images = [sample["rgb"].permute(1, 2, 0).numpy(), sample["sparse"][0].numpy(), sample["gt"][0].numpy()]
    if "teacher" in sample:
        images += [sample["teacher"][0].numpy(), sample["confidence"][0].numpy()]
    fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 3))
    for ax, name, values in zip(axes, names, images):
        if name == "RGB":
            ax.imshow(values)
        else:
            ax.imshow(np.ma.masked_where(values <= 0, values), cmap="turbo", vmin=0, vmax=1 if name == "C_cm" else 80)
        ax.set_title(name); ax.axis("off")
    fig.suptitle(label + ": " + sample["sid"])
    plt.show()
del train_data, val_data, train_sample, val_sample

## 6. Warm-start smoke trên mẫu thật

Strict migration kiểm đủ **572.017 tham số v3**, không drop old key; chỉ cho phép thiếu capacity.* của v4. Khởi tạo **D_full khớp v3**, không chỉ D1.

Kiểm actual-data forward/loss/backward và sparse/teacher gate. Không yêu cầu sensor gate đã train phải quay về giá trị init.

In [ ]:
command("smoke")
smoke = json.loads((WORK / "smoke_report.json").read_text())
assert smoke["passed"] and smoke["migration"]["old_keys_dropped"] == 0
assert smoke["migration"]["parent_parameters_loaded"] == 572017
shutil.copy2(WORK / "smoke_report.json", RUN_ROOT / "smoke_report.json")
print(json.dumps(smoke, indent=2))
torch.cuda.empty_cache()

## 7. Train thêm15epoch

Lần đầu runner đo đủ400val với checkpoint khởi tạo, lưu initial_val_metrics.json (epoch-1), rồi train epoch0–14 (cumulative30–44).
Nếu mất Colab, chạy lại notebook cùng MODEL_NAME/config: resume last.pth, không cộng thêm15epoch.

Sau mỗi epoch sync checkpoint và log về RUN_DIR. Best so cả initial; best=-1 nghĩa là chưa cải thiện do fine-tune.

Để làm control: sau khi xong v4, đổi MODEL_NAME="v3" ở cell1 rồi Run all với **cùng mọi setting còn lại**. Tốn thêm15epoch riêng, không bắt buộc.

In [ ]:
command("train")
print("Train log:", RUN_DIR / "train_log.csv")
print("Best:", RUN_DIR / "best.pth")
print("Last/resume:", RUN_DIR / "last.pth")

## 8. Accuracy: parent / initial / best / matched-budget control nếu có

Tất cả chấm cùng400val/25.424.992pixel GT. D_full soft, D1 pre và D_hard cùng mask.
Stage native-resolution chỉ là diagnostic pooled-GT, không coi RMSE ở các scale là cùng target.

Nếu đã chạy cả v3-control và v4, notebook kiểm recipe_sha256 trước khi đưa vào bảng.

In [ ]:
command("evaluate")
import pandas as pd

parent = json.loads((CODE / "parent_val_metrics.json").read_text())
initial = json.loads((RUN_DIR / "initial_val_metrics.json").read_text())
best = json.loads((RUN_DIR / "val_metrics.json").read_text())
run_manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
entries = [
    ("v3 parent, completed 30 epochs (A100 FP16)", parent, "final"),
    (MODEL_NAME + " initial, same runtime", initial, "final"),
    (MODEL_NAME + " best soft", best, "final"),
    (MODEL_NAME + " best pre-fusion", best, "pre_anchor"),
    (MODEL_NAME + " best legacy hard", best, "legacy_hard_anchor"),
]
other_name = "v3" if MODEL_NAME == "v4" else "v4"
other_dir = DRIVE_RUNS / RUN_NAMES[other_name] / "metric_kd"
if (other_dir / "val_metrics.json").is_file() and (other_dir / "run_manifest.json").is_file():
    other_manifest = json.loads((other_dir / "run_manifest.json").read_text())
    assert other_manifest["recipe_sha256"] == run_manifest["recipe_sha256"], "Different recipe/data: cannot label model-only comparison"
    other_metrics = json.loads((other_dir / "val_metrics.json").read_text())
    entries.append((other_name + " matched-budget best", other_metrics, "final"))
else:
    print("No matched-budget other-model result yet; parent comparison includes extra-training effects.")
rows = []
for label, report, policy in entries:
    values = report[policy]
    score = values["all"]
    assert score["pixels"] == 25424992, "GT support changed"
    rows.append({"model/output": label, "RMSE m": score["rmse_m"], "MAE m": score["mae_m"],
                 "iRMSE km^-1": score["irmse_km_inv"], "edge RMSE m": values["edge"]["rmse_m"],
                 **{f"RMSE {key} m": values[key]["rmse_m"] for key in ("0-20","20-40","40-60","60-80","80-120")}})
comparison = pd.DataFrame(rows)
comparison.to_csv(RUN_DIR / "comparison_accuracy.csv", index=False)
display(comparison)
print("Best new epoch:", best["checkpoint_epoch"], "(-1 = no improvement over initialization)")
print("Same-runtime improvement:", initial["final"]["all"]["rmse_m"] - best["final"]["all"]["rmse_m"], "m")
print("Reached <0.7 m:", best["final"]["all"]["rmse_m"] < .7)
display(pd.DataFrame(best["stage_native_gt_metrics"]).T)

## 9. Curves, loss budget và dynamics

Hệ số loss không phải tỷ lệ đóng góp. Xem weighted loss thực tế, ba output RMSE, delta mới và sensor trust.

In [ ]:
log = pd.read_csv(RUN_DIR / "train_log.csv")
assert len(log) == 15 and log["epoch"].tolist() == list(range(15)), "Run chưa hoàn thành 15 epoch"
assert set(log["train_samples"]) == {1600} and set(log["val_samples"]) == {400}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for key in ("val_rmse_m", "val_pre_anchor_rmse_m", "val_legacy_hard_rmse_m"):
    axes[0,0].plot(log["cumulative_epoch_index"], log[key], label=key)
axes[0,0].axhline(initial["final"]["all"]["rmse_m"], color="gray", linestyle="--", label="initial soft")
axes[0,0].set_ylabel("Global RMSE (m)"); axes[0,0].legend(fontsize=8)
for key in ("0-20","20-40","40-60","60-80"):
    axes[0,1].plot(log["epoch"], log[f"val_rmse_{key}_m"], label=key)
axes[0,1].set_ylabel("Range RMSE (m)"); axes[0,1].legend()
weighted = [key for key in log if key.startswith("loss_weighted_")]
for key in weighted:
    axes[1,0].plot(log["epoch"], log[key].clip(lower=1e-8), label=key.removeprefix("loss_weighted_"))
axes[1,0].set_yscale("log"); axes[1,0].set_ylabel("Weighted loss"); axes[1,0].legend(fontsize=8, ncol=2)
for key in ("loss_sensor_gate_mean","loss_abs_delta4_mean","loss_abs_delta1_mean","capacity_projection_norm"):
    axes[1,1].plot(log["epoch"], log[key], label=key.removeprefix("loss_"))
axes[1,1].legend(fontsize=8); axes[1,1].set_ylabel("Gate / delta(m) / KD coefficient")
for ax in axes.flat:
    ax.set_xlabel("Epoch (cumulative on top-left)"); ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(RUN_DIR / "training_diagnostics.png", dpi=160)
plt.show()
budget = log[["epoch", "loss_total", *weighted]].copy()
for key in weighted:
    budget[key + "_pct"] = 100 * budget[key] / budget["loss_total"]
budget.to_csv(RUN_DIR / "loss_budget.csv", index=False)
display(budget.tail(3))

## 10. Test 1.000 ảnh anonymous

Xuất dự đoán từ best checkpoint. **Không có public GT**: đây là tạo ZIP để submit, không phải phép đo test RMSE.

In [ ]:
command("test")
test_report = json.loads((RUN_DIR / "test_report.json").read_text())
assert test_report["samples"] == 1000 and test_report["anonymous_no_gt"] is True
print("Submission:", RUN_DIR / "kitti_test_predictions.zip")
print(json.dumps(test_report, indent=2))

## 11. Profile v3 parent và model best trên cùng GPU

Mỗi phép đo chạy subprocess riêng, FP16/batch1/352×1216. Không so latency A100 với T4.
V4 thêm params ở low resolution nhưng MAC tăng đáng kể; quyết định deploy dựa vào latency/P95/VRAM thực, không chỉ parameter count.

Decoder v4 gọi children để inject feature nên component timing tách thành decoder.*. Đây là pass instrumented, không phải các phần có thể cộng để thay total latency.

In [ ]:
command("profile", ("--parent",))
command("profile")
new_profile = json.loads((RUN_DIR / "profile.json").read_text())
old_profile = json.loads((RUN_DIR / "parent_profile_same_device.json").read_text())
keys = ("device","torch","precision","batch","shape","channels_last","compiled_total")
comparable = all(new_profile.get(key) == old_profile.get(key) for key in keys)
assert comparable, "Profile settings changed between parent and candidate"
efficiency = pd.DataFrame([{
    "model": name, "device": p["device"], "torch": p["torch"],
    "params": p["total_parameters"], "G Conv/Linear MAC": p["total_conv_linear_macs"]/1e9,
    "median ms": p["median_ms"], "P95 ms": p["p95_ms"], "peak MiB": p["peak_cuda_allocated_mib"]
} for name, p in (("v3 parent same device", old_profile), (MODEL_NAME + " best", new_profile))])
efficiency.to_csv(RUN_DIR / "comparison_efficiency.csv", index=False)
display(efficiency)
print("Measured median latency change %:", 100*(new_profile["median_ms"]/old_profile["median_ms"]-1))
components = pd.DataFrame({
    "parameters": new_profile["parameters"], "conv_linear_macs": new_profile["conv_linear_macs"],
    "instrumented_median_ms": new_profile["eager_component_median_ms"]
})
components.to_csv(RUN_DIR / "component_profile.csv")
display(components)

## 12. Export ONNX và tổng kết

Export **best đã chọn của run này**, static batch1/352×1216, opset17, FP32. Tự chạy checker và CPU ONNX Runtime parity cho sparse/empty-sparse. Đây chưa phải TensorRT engine.

In [ ]:
if RUN_EXPORT:
    command("export")
    export_report = json.loads((RUN_DIR / "export_report.json").read_text())
    assert export_report["onnxruntime_cpu_parity"] and not export_report["untrained_weights"]
    print(json.dumps(export_report, indent=2))
summary = {
    "model_name": MODEL_NAME, "recipe_sha256": run_manifest["recipe_sha256"], "architecture": cfg["architecture"], "parent_epoch": cfg["parent_epoch"],
    "extra_epochs_completed": len(log), "best_new_epoch": best["checkpoint_epoch"],
    "initial_soft_rmse_m": initial["final"]["all"]["rmse_m"],
    "best_soft_rmse_m": best["final"]["all"]["rmse_m"],
    "best_pre_fusion_rmse_m": best["pre_anchor"]["all"]["rmse_m"],
    "best_legacy_hard_rmse_m": best["legacy_hard_anchor"]["all"]["rmse_m"],
    "goal_0_7_reached": best["final"]["all"]["rmse_m"] < .7,
    "params": new_profile["total_parameters"], "gpu": new_profile["device"],
    "median_ms": new_profile["median_ms"], "runtime_settings_comparable_to_parent": comparable,
    "teacher_train_only": True, "anonymous_test_has_public_gt": False,
}
(RUN_DIR / "experiment_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("ALL OUTPUTS:", RUN_DIR)

## Output trên Drive

```text
MyDrive/GeoLift_RT_Runs/
├── AnchorFlow_v4_ContextWide_FT15/
│   ├── source_bundle/  # model v3 + v4, shared code, init checkpoint
│   ├── data_contract.json / smoke_report.json
│   └── metric_kd/
│       ├── initial_val_metrics.json / migration_report.json
│       ├── best.pth / last.pth
│       ├── train_log.csv / train_log.jsonl / train.log
│       ├── val_metrics.json / best_val_metrics.json
│       ├── run_manifest.json / resolved_config.json
│       ├── comparison_accuracy.csv / comparison_efficiency.csv
│       ├── parent_profile_same_device.json / profile.json
│       ├── component_profile.csv / loss_budget.csv
│       ├── training_diagnostics.png
│       ├── kitti_test_predictions.zip / test_report.json
│       ├── anchorflow_edge_fp32.onnx / export_report.json
│       └── experiment_summary.json
└── AnchorFlow_v3_Control_FT15/  # chỉ xuất hiện nếu bạn tự chọn MODEL_NAME="v3"
```

Test anonymous không public GT, không có test RMSE tại local. Model-only comparison cần **hai run cùng extra15epoch**, cùng recipe hash và nên cùng GPU. Không gọi gain v4-v3 parent là contribution riêng của capacity khi thiếu control.